# Phát hiện Deepfake bằng CNN + Multi-Head Self-Attention**Notebook chạy một lượt từ đầu đến cuối trên Kaggle.** Tái hiện bài báo*Attention-Enhanced CNN for High-Performance Deepfake Detection* (IEEE Access, vol. 13, 2025).---## Trước khi chạy — làm đúng 3 việc này**1. Bật GPU.** Panel bên phải → `Settings` → `Accelerator` → chọn **GPU T4 x2**.Không bật thì train sẽ mất hàng chục giờ thay vì khoảng 1 giờ.**2. Thêm dữ liệu.** Panel bên phải → `Input` → `+ Add Input` → tìm`140k-real-and-fake-faces` → bấm dấu **+** để thêm.**3. Bật Internet** (chỉ cần nếu bạn dùng phần Celeb-DF): `Settings` → `Internet` → `On`.Sau đó bấm **Run All**. Toàn bộ notebook mất khoảng **60–90 phút** với cấu hình mặc định.---## Notebook này làm những gì| Phần | Nội dung | Thời gian ||---|---|---|| 1 | Kiểm tra môi trường, cố định seed | 1 phút || 2 | Cấu hình — **chỗ duy nhất bạn cần sửa** | — || 3 | Nạp dữ liệu, cân bằng lớp, tăng cường dữ liệu | 2 phút || 4 | Dựng mô hình CNN + MHSA, kiểm tra đúng 696.001 tham số | 1 phút || 5 | Huấn luyện | 40–60 phút || 6 | Tìm ngưỡng tối ưu, ma trận nhầm lẫn, ROC/AUC | 3 phút || 7 | Grad-CAM — mô hình nhìn vào đâu | 2 phút || 8 | **Trọng số MHSA thật** — phần cốt lõi của đề tài | 3 phút || 9 | **Ablation** CNN thuần vs CNN+MHSA — chứng minh MHSA có ích | 40–60 phút || 10 | Tổng hợp kết quả, danh sách hình cần tải về | 1 phút |Mọi hình ảnh và bảng số liệu được lưu vào `/kaggle/working/ket_qua/` để bạn tải về đưa vào báo cáo.> **Quy ước nhãn, giữ nguyên xuyên suốt:** `fake = 0`, `real = 1`.> Đầu ra sigmoid là **xác suất ảnh là THẬT** (`p_real`). Đừng đổi giữa đường, sẽ sai hết kết quả.

---## Phần 1 — Kiểm tra môi trườngCố định seed để lần chạy sau ra kết quả giống lần này. Điều này **bắt buộc** chothí nghiệm ablation ở Phần 9 — nếu không cố định seed, so sánh có/không MHSA sẽkhông công bằng và hội đồng sẽ chỉ ra ngay.

In [ ]:
import os, random, json, time, globimport numpy as npimport tensorflow as tfSEED = 42os.environ["PYTHONHASHSEED"] = str(SEED)random.seed(SEED)np.random.seed(SEED)tf.random.set_seed(SEED)print("TensorFlow :", tf.__version__)print("Keras      :", tf.keras.__version__)gpus = tf.config.list_physical_devices("GPU")print("GPU tìm thấy:", len(gpus))for g in gpus:    print("  -", g.name)if not gpus:    print("\n*** CHƯA BẬT GPU ***")    print("Settings → Accelerator → GPU T4 x2, rồi chạy lại từ đầu.")else:    print("\nOK, có GPU.")# Xem dữ liệu đã được thêm chưaprint("\nDữ liệu trong /kaggle/input:")for d in sorted(os.listdir("/kaggle/input")) if os.path.exists("/kaggle/input") else []:    print("  -", d)

---## Phần 2 — Cấu hình**Đây là ô duy nhất bạn cần sửa.** Mọi phần sau đều đọc từ đây.Giải thích vài tham số quan trọng:- `SO_ANH_TRAIN`: số ảnh mỗi lớp dùng để train. Mặc định 10.000 mỗi lớp  (tổng 20.000) để chạy nhanh. Nếu có nhiều thời gian GPU, tăng lên 50.000  để tiến gần kết quả bài báo hơn. Đặt `None` để dùng toàn bộ dữ liệu.- `EPOCHS`: số lần mô hình học hết toàn bộ dữ liệu. 25 là đủ với dữ liệu rút gọn.- `BATCH`: số ảnh xử lý cùng lúc. Nếu báo lỗi hết bộ nhớ GPU (OOM), giảm xuống 32.- `LR` (learning rate): tốc độ học. 1e-4 theo đúng bài báo. Đừng tăng, dễ phân kỳ.- `CHAY_ABLATION`: bật Phần 9. Nó train thêm một mô hình nữa nên **tốn gấp đôi  thời gian**. Nhưng với đề tài về MHSA thì phần này rất nên có.

In [ ]:
# ---------------- ĐƯỜNG DẪN DỮ LIỆU ----------------# Cấu trúc tập 140k trên Kaggle: .../train|valid|test/real|fake/*.jpgGOC = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"# ---------------- KÍCH THƯỚC DỮ LIỆU ----------------SO_ANH_TRAIN = 10000   # mỗi lớp. Đặt None để dùng hếtSO_ANH_VALID = 2000    # mỗi lớpSO_ANH_TEST  = 2000    # mỗi lớp# ---------------- SIÊU THAM SỐ (theo bài báo) ----------------ANH    = 224     # kích thước ảnh vàoBATCH  = 64LR     = 1e-4EPOCHS = 25      # bài báo dùng 50ALPHA  = 0.25    # Focal LossGAMMA  = 2.0     # Focal LossPATIENCE_STOP   = 7PATIENCE_REDUCE = 3# ---------------- TÙY CHỌN ----------------CHAY_ABLATION = True    # Phần 9: train thêm mô hình CNN thuần để so sánhTIEU_CHI_NGUONG = "gmean"   # "gmean" hoặc "f1"# ---------------- NƠI LƯU KẾT QUẢ ----------------OUT = "/kaggle/working/ket_qua"os.makedirs(OUT, exist_ok=True)print("Thư mục dữ liệu tồn tại:", os.path.exists(GOC))if os.path.exists(GOC):    for split in ["train", "valid", "test"]:        for lop in ["real", "fake"]:            d = os.path.join(GOC, split, lop)            n = len(os.listdir(d)) if os.path.exists(d) else 0            print(f"  {split}/{lop}: {n:,} ảnh")else:    print("\n*** KHÔNG TÌM THẤY DỮ LIỆU ***")    print("Input → + Add Input → tìm '140k-real-and-fake-faces' → bấm +")

---## Phần 3 — Nạp và chuẩn bị dữ liệuBa việc diễn ra ở đây:**1. Cân bằng lớp.** Ta lấy **đúng bằng nhau** số ảnh real và fake. Nếu một lớpnhiều hơn hẳn, mô hình sẽ học cách đoán bừa lớp đó và vẫn có accuracy cao —một cái bẫy kinh điển.**2. Chuẩn hóa.** Chia mọi giá trị pixel cho 255 để đưa từ khoảng 0–255 về 0–1.Mạng nơ-ron học ổn định hơn nhiều với số nhỏ.**3. Tăng cường dữ liệu (augmentation).** Mỗi lần đưa ảnh vào train, ta xoay, dịch,phóng to, lật, đổi độ sáng một chút một cách ngẫu nhiên. Mô hình không bao giờthấy đúng cùng một ảnh hai lần, nên khó **học thuộc lòng** (overfitting) hơn.Chỉ áp dụng cho tập train — tập valid và test phải giữ nguyên để đánh giá công bằng.> **Lưu ý kỹ thuật:** Kaggle dùng TensorFlow 2.19 (Keras 3) đã **xóa**> `ImageDataGenerator`. Vì vậy notebook này dùng `tf.data` cùng các lớp> `layers.Random*` — cách làm hiện đại và nhanh hơn. Các file `.py` trong project> ghim TensorFlow 2.15 nên vẫn dùng `ImageDataGenerator`; hai cách cho kết quả tương đương.

In [ ]:
from tensorflow.keras import layersLOP = ["fake", "real"]   # thứ tự này quyết định fake=0, real=1AUTO = tf.data.AUTOTUNEdef lay_danh_sach(split, gioi_han):    # Trả về danh sách đường dẫn ảnh và nhãn, đã cân bằng hai lớp.    duong_dan, nhan = [], []    for nhan_so, lop in enumerate(LOP):          # fake->0, real->1        files = sorted(glob.glob(os.path.join(GOC, split, lop, "*.jpg")))        files += sorted(glob.glob(os.path.join(GOC, split, lop, "*.png")))        random.Random(SEED).shuffle(files)        if gioi_han is not None:            files = files[:gioi_han]        duong_dan += files        nhan += [nhan_so] * len(files)    return duong_dan, nhandef doc_anh(path, label):    img = tf.io.read_file(path)    img = tf.image.decode_jpeg(img, channels=3)    img = tf.image.resize(img, (ANH, ANH))    img = img / 255.0                              # chuẩn hóa về [0,1]    return img, tf.cast(label, tf.float32)# Các phép tăng cường, đúng theo bài báotang_cuong = tf.keras.Sequential([    layers.RandomFlip("horizontal"),    layers.RandomRotation(30 / 360),                  # +/- 30 độ    layers.RandomTranslation(0.2, 0.2),               # dịch 20%    layers.RandomZoom(0.2),    layers.RandomBrightness(0.1, value_range=(0.0, 1.0)),], name="tang_cuong")def tao_luong(split, gioi_han, huan_luyen):    duong_dan, nhan = lay_danh_sach(split, gioi_han)    ds = tf.data.Dataset.from_tensor_slices((duong_dan, nhan))    if huan_luyen:        ds = ds.shuffle(len(duong_dan), seed=SEED, reshuffle_each_iteration=True)    ds = ds.map(doc_anh, num_parallel_calls=AUTO).batch(BATCH)    if huan_luyen:        ds = ds.map(lambda x, y: (tang_cuong(x, training=True), y),                    num_parallel_calls=AUTO)    return ds.prefetch(AUTO), np.array(nhan)ds_train, y_train = tao_luong("train", SO_ANH_TRAIN, True)ds_valid, y_valid = tao_luong("valid", SO_ANH_VALID, False)ds_test,  y_test  = tao_luong("test",  SO_ANH_TEST,  False)for ten, y in [("train", y_train), ("valid", y_valid), ("test", y_test)]:    print(f"{ten:6s}: {len(y):6,} ảnh  |  fake {int((y==0).sum()):,}  real {int((y==1).sum()):,}")

Xem thử vài ảnh để chắc dữ liệu nạp đúng. Nếu ảnh bị méo màu hoặc nhãn lẫn lộn,hãy sửa ngay ở đây — đừng train rồi mới phát hiện.

In [ ]:
import matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltx_xem, y_xem = next(iter(ds_test))fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))for i, ax in enumerate(axes.ravel()):    ax.imshow(x_xem[i].numpy())    ax.set_title(LOP[int(y_xem[i].numpy())], fontsize=10)    ax.axis("off")fig.suptitle("Mẫu ảnh từ tập test (đã chuẩn hóa về [0,1])")fig.tight_layout()fig.savefig(f"{OUT}/00_mau_du_lieu.png", dpi=140)plt.show()print("Khoảng giá trị pixel:", float(x_xem.numpy().min()), "->", float(x_xem.numpy().max()))

---## Phần 4 — Mô hình CNN + MHSA### Luồng dữ liệu```Ảnh 224 x 224 x 3   |   |  4 khối tích chập. Mỗi khối: Conv3x3 -> BatchNorm -> ReLU -> MaxPool2x2 -> Dropout 0,25   |  Số kênh: 32 -> 64 -> 128 -> 256   |  Kích thước: 224 -> 112 -> 56 -> 28 -> 14   vBản đồ đặc trưng 14 x 14 x 256   |   |  Reshape: biến lưới 14x14 thành CHUỖI 196 ô, mỗi ô 256 chiều   vChuỗi 196 x 256   |   |  Dropout 0,3 -> MultiHeadAttention(4 head, key_dim 64) -> cộng residual -> LayerNorm   |  Đây là chỗ MỌI Ô SO SÁNH VỚI MỌI Ô   vChuỗi 196 x 256 (đã giàu thông tin toàn cục)   |   |  GlobalAveragePooling1D: gộp 196 ô thành 1 vector   vVector 256 chiều   |   |  Dense128 -> BN -> Dropout0,3 -> Dense64 -> BN -> Dropout0,3   vDense(1) sigmoid  ->  p_real (xác suất ảnh là THẬT)```### Hai điểm cốt lõi của đề tài**Vì sao cần attention?** Tích chập 3×3 chỉ nhìn 9 điểm lân cận, nên nó khó**so sánh** hai vùng cách xa nhau. Nhưng deepfake thường lộ ra đúng ở đó: mắt tráinét mắt phải mờ, da má mịn bất thường nhưng viền hàm gồ ghề, ánh sáng trên tránlệch với trên cổ. MHSA cho cả 196 ô so sánh đôi một nên bắt được loại bất nhất này.**Vì sao là `X + MHA(X)` chứ không phải chỉ `MHA(X)`?** Dấu cộng gọi là **kết nốitắt (residual)**. Nó khiến attention chỉ *bổ sung* thông tin vào đặc trưng CNN gốcthay vì thay thế, đồng thời cho gradient một đường đi thẳng về các tầng dưới,chống hiện tượng gradient tiêu biến.### Focal Loss"Loss" là điểm phạt: đoán sai bị phạt nhiều, mô hình tự điều chỉnh để giảm phạt.Vấn đề của loss thường là phần lớn ảnh quá dễ, nhưng chúng vẫn chiếm phần lớn tổngđiểm phạt nên mô hình lười học ảnh khó. Focal Loss nhân thêm hệ số $(1-p_t)^\gamma$:ảnh đã đoán đúng chắc chắn thì hệ số gần 0, gần như bị loại khỏi tổng phạt, buộcmô hình tập trung vào ảnh khó.

In [ ]:
from tensorflow.keras import models, regularizersdef focal_loss(alpha=ALPHA, gamma=GAMMA):    def ham(y_true, y_pred):        y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), tf.float32)        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)        a_t = y_true * alpha + (1 - y_true) * (1 - alpha)        return tf.reduce_mean(a_t * tf.pow(1 - p_t, gamma) * (-tf.math.log(p_t)))    ham.__name__ = "focal_loss"    return hamdef khoi_conv(x, so_filter, ten):    x = layers.Conv2D(so_filter, 3, padding="same",                      kernel_initializer="he_normal",                      kernel_regularizer=regularizers.l2(1e-4),                      name=f"{ten}_conv")(x)    x = layers.BatchNormalization(name=f"{ten}_bn")(x)    x = layers.Activation("relu", name=f"{ten}_relu")(x)    x = layers.MaxPooling2D(2, name=f"{ten}_pool")(x)    x = layers.Dropout(0.25, name=f"{ten}_drop")(x)    return xdef tao_mo_hinh(dung_mhsa=True):    # dung_mhsa=False -> bỏ khối attention, CHỈ dùng cho ablation ở Phần 9.    inp = layers.Input((ANH, ANH, 3), name="anh_vao")    x = khoi_conv(inp, 32,  "block1")   # 224 -> 112    x = khoi_conv(x,   64,  "block2")   # 112 -> 56    x = khoi_conv(x,   128, "block3")   # 56  -> 28    x = khoi_conv(x,   256, "block4")   # 28  -> 14    seq = layers.Reshape((14 * 14, 256), name="thanh_chuoi")(x)    if dung_mhsa:        a = layers.Dropout(0.3, name="drop_truoc_attn")(seq)        a = layers.MultiHeadAttention(num_heads=4, key_dim=64, dropout=0.2,                                      name="mhsa")(a, a)        x = layers.Add(name="cong_residual")([seq, a])      # X + MHA(X)        x = layers.LayerNormalization(name="layer_norm")(x)    else:        x = seq    x = layers.GlobalAveragePooling1D(name="gap")(x)    x = layers.Dense(128, activation="relu", name="dense_128")(x)    x = layers.BatchNormalization(name="bn_128")(x)    x = layers.Dropout(0.3, name="drop_128")(x)    x = layers.Dense(64, activation="relu", name="dense_64")(x)    x = layers.BatchNormalization(name="bn_64")(x)    x = layers.Dropout(0.3, name="drop_64")(x)    out = layers.Dense(1, activation="sigmoid", name="p_real")(x)    return models.Model(inp, out, name="CNN_MHSA" if dung_mhsa else "CNN_thuan")model = tao_mo_hinh(dung_mhsa=True)model.summary()tong = model.count_params()print("\nTổng tham số:", f"{tong:,}")if tong == 696001:    print("ĐÚNG — khớp chính xác con số 696.001 của bài báo.")else:    print(f"*** SAI *** Phải là 696.001, đang lệch {tong - 696001:+,}.")    print("Kiểm tra: num_heads=4, key_dim=64, Dense 128 và 64, chuỗi 196x256.")

### Ngân sách tham số — con số nên nhớ khi bảo vệÔ dưới đây tính phần đóng góp của từng khối. Kết quả: khối MHSA chiếm**263.168 tham số, tức 37,8% toàn mô hình**. Nêu được con số này cho thấy bạnhiểu mô hình mình xây chứ không phải sao chép.

In [ ]:
import pandas as pdnhom = {"4 khối Conv": 0, "Khối MHSA": 0, "LayerNorm": 0,        "Các tầng Dense": 0}for l in model.layers:    n = sum(int(np.prod(w.shape)) for w in l.weights)    if n == 0:        continue    if l.name.startswith("block"):        nhom["4 khối Conv"] += n    elif l.name == "mhsa":        nhom["Khối MHSA"] += n    elif l.name == "layer_norm":        nhom["LayerNorm"] += n    else:        nhom["Các tầng Dense"] += nbang_ts = pd.DataFrame({"Thành phần": list(nhom), "Số tham số": list(nhom.values())})bang_ts["Tỉ lệ %"] = (bang_ts["Số tham số"] / tong * 100).round(2)bang_ts.loc[len(bang_ts)] = ["TỔNG", tong, 100.0]display(bang_ts)bang_ts.to_csv(f"{OUT}/bang_tham_so.csv", index=False)print(f"\nKích thước mô hình: {tong * 4 / 1024**2:.2f} MB (float32)")print("So sánh: ResNet50 khoảng 25 triệu tham số, ViT-Base khoảng 86 triệu.")

---## Phần 5 — Huấn luyệnBốn cơ chế tự động chạy trong lúc train:- **ModelCheckpoint** — lưu lại mô hình ở epoch có `val_auc` cao nhất, không phải  epoch cuối. Quan trọng vì epoch cuối thường đã overfit.- **EarlyStopping** — nếu 7 epoch liền `val_auc` không cải thiện thì dừng sớm,  khỏi tốn giờ GPU.- **ReduceLROnPlateau** — nếu 3 epoch liền `val_loss` không giảm thì giảm một nửa  tốc độ học, giúp mô hình "hạ cánh" mượt hơn vào điểm tối ưu.- **CSVLogger** — ghi lịch sử ra file để bạn vẽ lại đồ thị cho báo cáo.> **Kaggle giới hạn 30 giờ GPU mỗi tuần và mỗi phiên tối đa 12 giờ.** Nếu bị ngắt> giữa lúc train, file `best.keras` trong `/kaggle/working/` vẫn còn — tải về, rồi> nạp lại bằng `tf.keras.models.load_model(..., custom_objects={"focal_loss": focal_loss()})`> để train tiếp.

In [ ]:
from tensorflow.keras.callbacks import (CSVLogger, EarlyStopping,                                        ModelCheckpoint, ReduceLROnPlateau)def bien_dich(m):    m.compile(optimizer=tf.keras.optimizers.Adam(LR),              loss=focal_loss(),              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])    return mdef huan_luyen(m, ten, epochs=EPOCHS):    thu_muc = f"/kaggle/working/runs_{ten}"    os.makedirs(thu_muc, exist_ok=True)    cb = [        ModelCheckpoint(f"{thu_muc}/best.keras", monitor="val_auc", mode="max",                        save_best_only=True, verbose=1),        EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE_STOP,                      restore_best_weights=True, verbose=1),        ReduceLROnPlateau(monitor="val_loss", factor=0.5,                          patience=PATIENCE_REDUCE, min_lr=1e-7, verbose=1),        CSVLogger(f"{thu_muc}/history.csv"),    ]    t0 = time.time()    hist = m.fit(ds_train, validation_data=ds_valid, epochs=epochs,                 callbacks=cb, verbose=1)    print(f"\nHuấn luyện xong sau {(time.time()-t0)/60:.1f} phút.")    return m, histmodel = bien_dich(model)model, hist = huan_luyen(model, "mhsa")

### Đọc đồ thị huấn luyệnĐây là hình **bắt buộc** phải có trong báo cáo. Cách đọc:- Hai đường train và valid đi gần nhau → mô hình học tốt, khái quát hóa được.- Đường train tiếp tục tốt lên nhưng valid đứng im hoặc xấu đi → **overfitting**,  mô hình đang học thuộc lòng tập train. Báo cáo BC-BTL gặp đúng chuyện này:  ở epoch 50, train accuracy 0,925 nhưng val accuracy chỉ 0,726.- Chỗ đường gấp khúc xuống → là lúc `ReduceLROnPlateau` giảm tốc độ học.

In [ ]:
def ve_lich_su(hist, ten):    h = hist.history    fig, axes = plt.subplots(1, 3, figsize=(15, 4))    for ax, khoa, tieu_de in zip(axes, ["loss", "accuracy", "auc"],                                 ["Focal Loss", "Accuracy", "AUC"]):        ax.plot(h[khoa], label="train", lw=2)        ax.plot(h["val_" + khoa], label="validation", lw=2)        ax.set_xlabel("Epoch")        ax.set_title(tieu_de)        ax.legend()        ax.grid(alpha=0.3)    fig.suptitle(f"Quá trình huấn luyện — {ten}")    fig.tight_layout()    fig.savefig(f"{OUT}/01_lich_su_{ten}.png", dpi=150)    plt.show()    i = int(np.argmax(h["val_auc"]))    print(f"Epoch tốt nhất: {i+1}  |  val_auc {h['val_auc'][i]:.4f}  "          f"|  val_accuracy {h['val_accuracy'][i]:.4f}")    kc = h["accuracy"][-1] - h["val_accuracy"][-1]    print(f"Chênh lệch train-val ở epoch cuối: {kc:+.4f}", end="  ")    print("(trên 0,10 là có overfitting, cần nêu trong báo cáo)" if kc > 0.10 else "(bình thường)")ve_lich_su(hist, "CNN+MHSA")

---## Phần 6 — Đánh giá và tìm ngưỡng tối ưu### Vì sao không dùng ngưỡng 0,5?Mô hình trả về một con số 0–1, ta cần một mốc để quyết định "trên mốc thì gọi làthật". Mặc định ai cũng lấy 0,5, nhưng Focal Loss với $\alpha = 0{,}25$ làm mô hình**lệch hẳn về một lớp**, nên ở mốc 0,5 kết quả rất tệ dù mô hình thực ra phân biệttốt.Cách chữa: quét **mọi** mốc có thể trên đường ROC rồi chọn mốc tốt nhất theo:$$\text{G-Mean} = \sqrt{TPR \times (1 - FPR)}$$G-Mean chỉ cao khi mô hình làm tốt **cả hai lớp cùng lúc**, nên nó không bị lừabởi trường hợp đoán bừa một lớp. Báo cáo BC-BTL tìm ra $\theta = 0{,}687$ cho tập140K và $\theta = 0{,}538$ cho Celeb-DF v2 bằng đúng cách này — và đó là đóng gópriêng của báo cáo đó so với bài báo gốc.### Các chỉ số cần hiểu| Chỉ số | Nghĩa | Phụ thuộc ngưỡng? ||---|---|---|| **AUC** | Khả năng phân biệt hai lớp. 1,0 hoàn hảo, 0,5 đoán bừa | **Không** || **Accuracy** | Tỉ lệ đoán đúng trên tổng số ảnh | Có || **Precision** | Trong các ảnh bị gọi là fake, bao nhiêu phần đúng là fake | Có || **Recall** | Trong các ảnh fake thực tế, bắt được bao nhiêu phần | Có || **F1** | Trung bình điều hòa của Precision và Recall | Có |AUC là chỉ số quan trọng nhất để so với bài báo, chính vì nó không phụ thuộc ngưỡng.

In [ ]:
import seaborn as snsfrom sklearn.metrics import (classification_report, confusion_matrix, f1_score,                             roc_auc_score, roc_curve)def tim_nguong(y, p, tieu_chi="gmean"):    fpr, tpr, nguongs = roc_curve(y, p)    if tieu_chi == "gmean":        diem = np.sqrt(tpr * (1 - fpr))    else:        diem = np.array([f1_score(y, (p >= t).astype(int), zero_division=0)                         for t in nguongs])    i = int(np.nanargmax(diem))    return float(nguongs[i]), float(diem[i])def danh_gia(m, ten, nguong=None, tieu_chi=TIEU_CHI_NGUONG):    p = m.predict(ds_test, verbose=1).ravel()      # p_real    y = y_test    auc = roc_auc_score(y, p)    if nguong is None:        nguong, diem = tim_nguong(y, p, tieu_chi)        print(f"\nNgưỡng tối ưu theo {tieu_chi}: θ = {nguong:.4f} (điểm {diem:.4f})")    else:        print(f"\nDùng ngưỡng cho trước: θ = {nguong:.4f}")    du_doan = (p >= nguong).astype(int)    acc = float((y == du_doan).mean())    acc_05 = float((y == (p >= 0.5).astype(int)).mean())    print(f"Accuracy ở θ tối ưu : {acc*100:.2f}%")    print(f"Accuracy ở θ = 0,5  : {acc_05*100:.2f}%   <- chênh lệch cho thấy vì sao phải tối ưu ngưỡng")    print(f"AUC                 : {auc:.4f}\n")    bao_cao = classification_report(y, du_doan, target_names=LOP,                                   digits=4, zero_division=0)    print(bao_cao)    cm = confusion_matrix(y, du_doan)    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,                xticklabels=LOP, yticklabels=LOP, ax=axes[0])    axes[0].set_xlabel("Dự đoán"); axes[0].set_ylabel("Thực tế")    axes[0].set_title(f"Ma trận nhầm lẫn (θ={nguong:.3f})")    fpr, tpr, _ = roc_curve(y, p)    axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")    axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Đoán ngẫu nhiên")    axes[1].set_xlabel("Tỉ lệ dương tính giả (FPR)")    axes[1].set_ylabel("Tỉ lệ dương tính thật (TPR)")    axes[1].set_title("Đường cong ROC")    axes[1].legend(loc="lower right")    fig.suptitle(f"Kết quả trên tập test — {ten}")    fig.tight_layout()    fig.savefig(f"{OUT}/02_danh_gia_{ten}.png", dpi=150)    plt.show()    kq = {"ten": ten, "nguong": nguong, "auc": float(auc), "accuracy": acc,          "accuracy_nguong_05": acc_05, "confusion_matrix": cm.tolist(),          "bao_cao": bao_cao}    with open(f"{OUT}/chi_so_{ten}.json", "w") as f:        json.dump(kq, f, indent=2, ensure_ascii=False)    np.save(f"{OUT}/p_{ten}.npy", p)    np.save(f"{OUT}/y_{ten}.npy", y)    return kq, pkq_mhsa, p_mhsa = danh_gia(model, "CNN_MHSA")

### So sánh với bài báo gốc và với báo cáo BC-BTLBảng dưới đây đối chiếu kết quả của bạn với hai mốc tham khảo. **Thấp hơn là bìnhthường và phải giải thích được lý do**, đó là phần ăn điểm chứ không phải phầnphải che giấu.Bốn lý do chính đáng để nêu: (1) bạn dùng ít dữ liệu và ít epoch hơn; (2) GPU T4trên Kaggle so với 2× RTX 4090 của bài báo; (3) bài báo dùng tập test rất nhỏ(khoảng 1,44% dữ liệu) nên số của họ dễ cao hơn, tập test của bạn lớn hơn nên đánhgiá khắt khe hơn; (4) khác biệt trong cách chia dữ liệu và tiền xử lý.

In [ ]:
so_sanh = pd.DataFrame([    {"Nguồn": "Bài báo gốc (IEEE Access 2025)", "Tập": "140K", "Accuracy %": 97.70, "AUC %": 99.75},    {"Nguồn": "Báo cáo BC-BTL", "Tập": "140K", "Accuracy %": 90.67, "AUC %": 96.55},    {"Nguồn": "Notebook này", "Tập": "140K",     "Accuracy %": round(kq_mhsa["accuracy"] * 100, 2),     "AUC %": round(kq_mhsa["auc"] * 100, 2)},])display(so_sanh)so_sanh.to_csv(f"{OUT}/so_sanh_voi_bai_bao.csv", index=False)print(f"Chênh so với bài báo gốc : {kq_mhsa['accuracy']*100 - 97.70:+.2f}% accuracy, "      f"{kq_mhsa['auc']*100 - 99.75:+.2f}% AUC")print(f"Chênh so với BC-BTL      : {kq_mhsa['accuracy']*100 - 90.67:+.2f}% accuracy, "      f"{kq_mhsa['auc']*100 - 96.55:+.2f}% AUC")print("\nLưu ý: con số của bài báo gốc và BC-BTL lấy từ hai tài liệu trong project,")print("chép lại để đối chiếu. Notebook này không kiểm chứng lại được chúng.")

---## Phần 7 — Grad-CAM: mô hình nhìn vào đâuGrad-CAM cho thấy vùng nào trên ảnh ảnh hưởng mạnh nhất tới quyết định. Năm bước:1. Lấy bản đồ đặc trưng 14×14×256 ở cuối khối Conv 4.2. Tính gradient của điểm dự đoán theo bản đồ đó — kênh nào ảnh hưởng mạnh.3. Lấy trung bình gradient trên toàn không gian làm **trọng số quan trọng** cho từng kênh.4. Cộng có trọng số 256 kênh lại thành bản đồ nhiệt 14×14, bỏ giá trị âm bằng ReLU.5. Chuẩn hóa về [0,1], phóng to lên 224×224 bằng nội suy song tuyến, phủ màu jet.Notebook xuất 4 trường hợp: fake nhận đúng, real nhận đúng, fake bị bỏ sót, vàreal bị báo động giả. **Hai trường hợp sai quan trọng hơn hai trường hợp đúng** —chúng cho bạn chất liệu để phân tích điểm yếu của mô hình trong báo cáo.> Grad-CAM áp dụng được cho **bất kỳ** CNN nào, kể cả mạng không có attention. Vì> vậy nó **không chứng minh điều gì về MHSA**. Phần 8 mới làm việc đó.

In [ ]:
TANG_CONV = "block4_pool"      # đầu ra 14x14x256grad_model = tf.keras.Model(model.input,                            [model.get_layer(TANG_CONV).output, model.output])def grad_cam(x, giai_thich="fake"):    with tf.GradientTape() as tape:        fmap, p = grad_model(x, training=False)        tape.watch(fmap)        diem = tf.reduce_sum(p if giai_thich == "real" else 1.0 - p)    g = tape.gradient(diem, fmap)                      # (b,14,14,256)    w = tf.reduce_mean(g, axis=(1, 2))                 # (b,256)    heat = tf.nn.relu(tf.einsum("bhwc,bc->bhw", fmap, w))[..., None]    mn = tf.reduce_min(heat, axis=[1, 2, 3], keepdims=True)    mx = tf.reduce_max(heat, axis=[1, 2, 3], keepdims=True)    heat = (heat - mn) / (mx - mn + 1e-8)    return tf.image.resize(heat, (ANH, ANH), method="bilinear").numpy(), p.numpy().ravel()def phu_mau(anh, heat, alpha=0.45):    mau = plt.get_cmap("jet")(heat[..., 0])[..., :3]    return np.clip((1 - alpha) * anh + alpha * mau, 0, 1)theta = kq_mhsa["nguong"]nhan_tr = {(0, 0): "Fake nhận ĐÚNG", (1, 1): "Real nhận ĐÚNG",           (0, 1): "Fake bị bỏ sót (nhận sai thành Real)",           (1, 0): "Real bị báo động giả (nhận sai thành Fake)"}gio = {k: [] for k in nhan_tr}CAN = 3for xb, yb in ds_test:    hf, p = grad_cam(xb, "fake")    hr, _ = grad_cam(xb, "real")    dd = (p >= theta).astype(int)    yb = yb.numpy().ravel().astype(int)    for i in range(len(yb)):        k = (int(yb[i]), int(dd[i]))        if len(gio[k]) < CAN:            gio[k].append((xb[i].numpy(), hr[i] if dd[i] == 1 else hf[i], p[i]))    if all(len(v) >= CAN for v in gio.values()):        breakfor k, items in gio.items():    if not items:        print("Không tìm thấy trường hợp:", nhan_tr[k], "(mô hình rất tốt ở nhóm này)")        continue    n = len(items)    fig, axes = plt.subplots(2, n, figsize=(3.3 * n, 6.8), squeeze=False)    for j, (img, heat, pr) in enumerate(items):        axes[0][j].imshow(img)        axes[0][j].set_title(f"Ảnh gốc (p_real={pr:.2f})", fontsize=9)        axes[0][j].axis("off")        axes[1][j].imshow(phu_mau(img, heat))        axes[1][j].set_title("Grad-CAM", fontsize=9)        axes[1][j].axis("off")    fig.suptitle(nhan_tr[k])    fig.tight_layout()    fig.savefig(f"{OUT}/03_gradcam_{k[0]}{k[1]}.png", dpi=150)    plt.show()

---## Phần 8 — Trọng số MHSA thật: phần cốt lõi của đề tàiĐây là phần **quan trọng nhất** vì đề tài của bạn là CNN + MHSA. Ta bóc trực tiếpma trận trọng số attention $A$ kích thước **196×196** ra từ tầng MHSA, đúng nhưbài báo gốc làm.**Cách đọc ma trận $A$:** $A[i][j]$ = "khi xử lý ô thứ $i$, mô hình dành bao nhiêuphần chú ý cho ô thứ $j$". 196 ô là lưới 14×14 trên khuôn mặt, mỗi ô tương ứngkhoảng 16×16 pixel ảnh gốc.Notebook xuất 3 hình và 2 số liệu:- **Hình 1** — độ chú ý *nhận về*: ô nào được tất cả các ô khác quan tâm nhiều nhất,  tức vùng mặt quan trọng nhất.- **Hình 2** — một ô *nhìn đi đâu*: chứng minh attention liên kết được các vùng  **cách xa nhau**, việc mà một tầng tích chập 3×3 không thể làm.- **Hình 3** — so sánh 4 head: mỗi head học một kiểu quan hệ khác nhau, giải thích  vì sao cần *multi*-head chứ không phải một head.- **Entropy attention**: tối đa là $\ln 196 = 5{,}278$, nghĩa là chú ý dàn trải đều  khắp — tức không học được gì. Đo được thấp hơn đáng kể là bằng chứng khối MHSA  đã học được sự tập trung có chọn lọc.- **Khoảng cách chú ý trung bình**: nếu lớn hơn khoảng 3 ô thì attention đang liên  kết các vùng vượt xa tầm với của tích chập 3×3. Đây là con số mạnh nhất để bảo vệ đề tài.

In [ ]:
LUOI = 14mhsa_layer = model.get_layer("mhsa")seq_out = model.get_layer("thanh_chuoi").outputa_in = model.get_layer("drop_truoc_attn")(seq_out, training=False)_, scores_out = mhsa_layer(a_in, a_in, return_attention_scores=True, training=False)attn_model = tf.keras.Model(model.input, [scores_out, model.output])xb, yb = next(iter(ds_test))N_XEM = 3xb, yb = xb[:N_XEM], yb[:N_XEM]scores, p = attn_model(xb, training=False)scores = scores.numpy()          # (b, 4 head, 196, 196)p = p.numpy().ravel()yb = yb.numpy().ravel().astype(int)print("Kích thước ma trận attention:", scores.shape,      "= (số ảnh, số head, 196 ô, 196 ô)")def overlay(ax, anh, ban_do, tieu_de):    h = ban_do - ban_do.min()    h = h / (h.max() + 1e-8)    h = tf.image.resize(h[..., None], (ANH, ANH), method="bilinear").numpy()[..., 0]    ax.imshow(anh); ax.imshow(h, cmap="jet", alpha=0.45)    ax.set_title(tieu_de, fontsize=9); ax.axis("off")# --- Hình 1: độ chú ý nhận về ---fig, axes = plt.subplots(2, N_XEM, figsize=(3.4 * N_XEM, 7), squeeze=False)for i in range(N_XEM):    axes[0][i].imshow(xb[i].numpy())    axes[0][i].set_title(f"{LOP[yb[i]]} (p_real={p[i]:.2f})", fontsize=9)    axes[0][i].axis("off")    nhan_ve = scores[i].mean(axis=0).mean(axis=0).reshape(LUOI, LUOI)    overlay(axes[1][i], xb[i].numpy(), nhan_ve, "Độ chú ý nhận về")fig.suptitle("Trọng số MHSA — vùng mặt được chú ý nhất")fig.tight_layout(); fig.savefig(f"{OUT}/04_attention_nhanve.png", dpi=150); plt.show()# --- Hình 2: một ô nhìn đi đâu ---O_NGUON = 98                      # ô giữa mặthang, cot = divmod(O_NGUON, LUOI)fig, axes = plt.subplots(1, N_XEM, figsize=(3.4 * N_XEM, 3.8), squeeze=False)for i in range(N_XEM):    di_den = scores[i].mean(axis=0)[O_NGUON].reshape(LUOI, LUOI)    overlay(axes[0][i], xb[i].numpy(), di_den, f"Ô ({hang},{cot}) chú ý tới đâu")    axes[0][i].plot((cot + 0.5) * ANH / LUOI, (hang + 0.5) * ANH / LUOI,                    "w*", markersize=15, markeredgecolor="black")fig.suptitle("Attention liên kết các vùng cách xa nhau (dấu sao = ô nguồn)")fig.tight_layout(); fig.savefig(f"{OUT}/05_attention_tu_mot_o.png", dpi=150); plt.show()# --- Hình 3: so sánh 4 head ---fig, axes = plt.subplots(1, 5, figsize=(16.5, 3.8))axes[0].imshow(xb[0].numpy())axes[0].set_title(f"Ảnh gốc ({LOP[yb[0]]})", fontsize=9); axes[0].axis("off")for h in range(4):    overlay(axes[h + 1], xb[0].numpy(),            scores[0][h].mean(axis=0).reshape(LUOI, LUOI), f"Head {h+1}")fig.suptitle("Bốn head học bốn kiểu quan hệ khác nhau — lý do cần Multi-Head")fig.tight_layout(); fig.savefig(f"{OUT}/06_attention_4head.png", dpi=150); plt.show()

In [ ]:
# --- Hai số liệu định lượng để đưa vào báo cáo ---A = scores.mean(axis=(0, 1))                 # (196,196), trung bình mọi ảnh và headent = float(-(A * np.log(A + 1e-12)).sum(axis=1).mean())ent_max = float(np.log(196))toa_do = np.stack(np.meshgrid(np.arange(LUOI), np.arange(LUOI), indexing="ij"), -1)toa_do = toa_do.reshape(-1, 2).astype(float)kc = np.linalg.norm(toa_do[:, None, :] - toa_do[None, :, :], axis=-1)kc_tb = float((A * kc).sum(axis=1).mean())bang_attn = pd.DataFrame([    {"Chỉ số": "Entropy attention", "Giá trị": round(ent, 4),     "Tối đa": round(ent_max, 4),     "Diễn giải": "Càng thấp so với tối đa thì attention càng tập trung có chọn lọc"},    {"Chỉ số": "Khoảng cách chú ý trung bình (ô)", "Giá trị": round(kc_tb, 4),     "Tối đa": 18.38,     "Diễn giải": "Lớn hơn 3 ô nghĩa là vượt xa tầm với của tích chập 3x3"},])display(bang_attn)bang_attn.to_csv(f"{OUT}/bang_chi_so_attention.csv", index=False)print(f"Entropy {ent:.4f} / {ent_max:.4f} tối đa  ->  ", end="")print("attention TẬP TRUNG có chọn lọc." if ent < ent_max - 0.3      else "attention còn dàn trải; cần train thêm epoch.")print(f"Khoảng cách chú ý {kc_tb:.2f} ô  ->  ", end="")print("attention liên kết vùng XA, đúng vai trò của nó." if kc_tb > 3      else "attention còn khá cục bộ; nêu đây là hạn chế trong báo cáo.")# Ma trận attention thô, để minh họa cấu trúcfig, ax = plt.subplots(figsize=(5.6, 4.8))im = ax.imshow(A, cmap="viridis")ax.set_xlabel("Ô được chú ý (j)"); ax.set_ylabel("Ô đang xử lý (i)")ax.set_title("Ma trận attention A (196×196), trung bình mọi head")fig.colorbar(im, ax=ax, label="Trọng số chú ý")fig.tight_layout(); fig.savefig(f"{OUT}/07_ma_tran_attention.png", dpi=150); plt.show()

---## Phần 9 — Ablation: chứng minh MHSA thực sự có íchCâu hỏi bảo vệ nguy hiểm nhất với đề tài của bạn: **"Em nói MHSA giúp cải thiện,vậy bằng chứng đâu?"**Bài báo gốc **không** làm thí nghiệm này. Nếu bạn làm, bạn có một đóng góp riêngvà trả lời được bằng số liệu thay vì bằng lý thuyết.**Cách làm:** train hai mô hình trên **cùng dữ liệu, cùng siêu tham số, cùng sốepoch, cùng seed** — chỉ khác duy nhất một điều là có hay không khối MHSA. Rồi so AUC.| Mô hình | Số tham số ||---|---|| CNN thuần (không attention) | 432.321 || CNN + MHSA (đề tài của bạn) | 696.001 |Phần này train thêm một mô hình nữa nên **tốn thêm 40–60 phút**. Nếu hết giờ GPU,đặt `CHAY_ABLATION = False` ở Phần 2 và chạy lại phần này trong một phiên khác.> **Lời khuyên thẳng thắn:** nếu kết quả cho thấy MHSA *không* cải thiện, hãy báo> cáo trung thực kèm giải thích nguyên nhân. Một đồ án trung thực với kết quả âm và> phân tích tốt được đánh giá cao hơn một đồ án có số liệu đẹp nhưng không giải> thích được. Khi đó bạn vẫn còn bằng chứng từ Phần 8.

In [ ]:
if CHAY_ABLATION:    # Cố định lại seed để hai lần train công bằng    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)    model_cnn = tao_mo_hinh(dung_mhsa=False)    print("CNN thuần :", f"{model_cnn.count_params():,} tham số")    print("CNN + MHSA:", f"{tong:,} tham số")    print("Chênh lệch:", f"{tong - model_cnn.count_params():,} tham số của khối MHSA\n")    model_cnn = bien_dich(model_cnn)    model_cnn, hist_cnn = huan_luyen(model_cnn, "cnn_thuan")    ve_lich_su(hist_cnn, "CNN thuần")    kq_cnn, p_cnn = danh_gia(model_cnn, "CNN_thuan")else:    kq_cnn = None    print("Đã bỏ qua ablation (CHAY_ABLATION = False).")

In [ ]:
if kq_cnn is not None:    bang_ab = pd.DataFrame([        {"Mô hình": "CNN thuần", "Số tham số": model_cnn.count_params(),         "θ": round(kq_cnn["nguong"], 4),         "Accuracy %": round(kq_cnn["accuracy"] * 100, 2),         "AUC %": round(kq_cnn["auc"] * 100, 2)},        {"Mô hình": "CNN + MHSA", "Số tham số": tong,         "θ": round(kq_mhsa["nguong"], 4),         "Accuracy %": round(kq_mhsa["accuracy"] * 100, 2),         "AUC %": round(kq_mhsa["auc"] * 100, 2)},    ])    display(bang_ab)    bang_ab.to_csv(f"{OUT}/bang_ablation.csv", index=False)    d_auc = (kq_mhsa["auc"] - kq_cnn["auc"]) * 100    d_acc = (kq_mhsa["accuracy"] - kq_cnn["accuracy"]) * 100    print(f"\nMHSA làm AUC thay đổi      : {d_auc:+.2f} điểm phần trăm")    print(f"MHSA làm Accuracy thay đổi : {d_acc:+.2f} điểm phần trăm")    print(f"Giá phải trả                : +{(tong/model_cnn.count_params()-1)*100:.1f}% tham số\n")    if d_auc > 0.5:        print("KẾT LUẬN: MHSA cải thiện rõ rệt. Đây là bằng chứng định lượng cho")        print("đề tài của bạn — đưa bảng và hình này vào Chương 3 báo cáo.")    elif d_auc > -0.5:        print("KẾT LUẬN: chênh lệch không đáng kể. ĐỪNG che giấu.")        print("Giải thích: với lượng dữ liệu và số epoch hạn chế, lợi ích của")        print("attention chưa bộc lộ. Nêu là hạn chế trong phần Kết luận, và dùng")        print("kết quả Phần 8 để chứng minh khối MHSA vẫn học được cấu trúc có nghĩa.")    else:        print("KẾT LUẬN: mô hình có MHSA kém hơn. Kiểm tra theo thứ tự:")        print("(a) hai lần train có cùng số epoch và cùng seed không,")        print("(b) mô hình lớn hơn có bị overfit trên dữ liệu nhỏ không,")        print("(c) dropout 0,3 trước attention có quá mạnh không.")        print("Xử lý (a) trước khi kết luận bất cứ điều gì.")    # Hình cột và ROC chồng nhau    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))    nhan_mh = ["CNN thuần", "CNN + MHSA"]    mau = ["#9aa4b2", "#2563eb"]    for ax, khoa, tieu_de in zip(axes[:2], ["accuracy", "auc"], ["Accuracy", "AUC"]):        gt = [kq_cnn[khoa], kq_mhsa[khoa]]        ax.bar(nhan_mh, gt, color=mau, width=0.55)        for i, v in enumerate(gt):            ax.text(i, v + 0.008, f"{v:.4f}", ha="center", fontsize=10)        ax.set_ylim(0, max(1.0, max(gt) + 0.08))        ax.set_title(tieu_de); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)    for pp, nh, m, kq in [(p_cnn, "CNN thuần", mau[0], kq_cnn),                          (p_mhsa, "CNN + MHSA", mau[1], kq_mhsa)]:        fpr, tpr, _ = roc_curve(y_test, pp)        axes[2].plot(fpr, tpr, color=m, lw=2, label=f"{nh} (AUC={kq['auc']:.4f})")    axes[2].plot([0, 1], [0, 1], "k--", lw=1, label="Đoán ngẫu nhiên")    axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")    axes[2].set_title("So sánh ROC"); axes[2].legend(loc="lower right", fontsize=9)    fig.suptitle("Thí nghiệm loại bỏ: đóng góp của khối MHSA")    fig.tight_layout(); fig.savefig(f"{OUT}/08_ablation.png", dpi=150); plt.show()

---## Phần 10 — Tổng hợp và tải kết quả vềÔ dưới in ra toàn bộ số liệu cùng danh sách file đã tạo. Tải chúng về từ tab**Output** ở panel bên phải.

In [ ]:
print("=" * 62)print("TỔNG HỢP KẾT QUẢ")print("=" * 62)print(f"Số ảnh train / valid / test : {len(y_train):,} / {len(y_valid):,} / {len(y_test):,}")print(f"Số epoch thực tế đã chạy    : {len(hist.history['loss'])} (đặt tối đa {EPOCHS})")print(f"Tham số CNN + MHSA          : {tong:,}")print()print(f"Ngưỡng tối ưu θ             : {kq_mhsa['nguong']:.4f}")print(f"Accuracy tại θ tối ưu       : {kq_mhsa['accuracy']*100:.2f}%")print(f"Accuracy tại θ = 0,5        : {kq_mhsa['accuracy_nguong_05']*100:.2f}%")print(f"AUC                         : {kq_mhsa['auc']:.4f}")print()print(f"Entropy attention           : {ent:.4f} / {ent_max:.4f} tối đa")print(f"Khoảng cách chú ý trung bình: {kc_tb:.2f} ô")if kq_cnn is not None:    print()    print(f"Ablation — AUC không MHSA   : {kq_cnn['auc']:.4f}")    print(f"Ablation — AUC có MHSA      : {kq_mhsa['auc']:.4f}")    print(f"Đóng góp của MHSA           : {(kq_mhsa['auc']-kq_cnn['auc'])*100:+.2f} điểm AUC")print("=" * 62)print("\nFile đã tạo trong", OUT, ":")for f in sorted(os.listdir(OUT)):    kb = os.path.getsize(os.path.join(OUT, f)) / 1024    print(f"  {f:38s} {kb:8.1f} KB")print("\nMô hình đã lưu:")for d in sorted(glob.glob("/kaggle/working/runs_*/best.keras")):    print("  ", d, f"({os.path.getsize(d)/1024**2:.2f} MB)")

---## Hình nào dùng vào đâu trong báo cáo| File | Mục trong báo cáo ||---|---|| `00_mau_du_lieu.png` | Ch.2 — Thu thập và tiền xử lý dữ liệu || `bang_tham_so.csv` | Ch.2 — Tổng quan tham số mô hình || `01_lich_su_*.png` | Ch.3 — Quá trình huấn luyện, phân tích overfitting || `02_danh_gia_*.png` | Ch.3 — Ma trận nhầm lẫn và đường ROC || `so_sanh_voi_bai_bao.csv` | Ch.3 — So sánh với bài báo gốc || `03_gradcam_*.png` | Ch.3 — Khả năng giải thích của mô hình || `04`–`07_attention_*.png` | Ch.3 — **Phân tích cơ chế MHSA** (phần cốt lõi) || `bang_chi_so_attention.csv` | Ch.3 — Số liệu định lượng về attention || `08_ablation.png`, `bang_ablation.csv` | Ch.3 — **Thí nghiệm loại bỏ, đóng góp riêng** |## Việc tiếp theo**1. Tải mọi thứ về ngay.** `/kaggle/working/` bị xóa khi phiên kết thúc.Tab Output ở panel bên phải, tải cả thư mục `ket_qua` và các file `best.keras`.**2. Làm tập dữ liệu thứ hai (Celeb-DF v2).** Xin quyền truy cập từ tác giả — làmviệc này **sớm** vì có thể mất vài ngày chờ. Sau đó chạy `buoc1_trich_xuat_mat.py`để tách mặt bằng MediaPipe, rồi `buoc1b_chia_du_lieu.py` để chia dữ liệu**theo video** (nếu chia theo ảnh, các frame cùng một video rơi vào cả train vàtest, điểm số bị thổi phồng giả tạo).**3. Đánh giá chéo hai tập.** Đây là phát hiện quan trọng nhất của bài báo:accuracy sẽ tụt về khoảng 35–50% vì mô hình học *dấu vết riêng của một kỹ thuậtsinh ảnh* thay vì học *thế nào là mặt giả nói chung*. Đây không phải lỗi của bạn.## Nguồn- S. Dasgupta, J. Mason, X. Yuan, O. Odeyomi, K. Roy, "Attention-Enhanced CNN for  High-Performance Deepfake Detection: A Multi-Dataset Study", *IEEE Access*,  vol. 13, 2025, tr. 101980–101993. (PDF trong project)- Báo cáo BC-BTL (trong project) — nguồn của cách làm tối giản và của các mốc  90,67% / 96,55%.- A. Vaswani và cộng sự, "Attention Is All You Need", NeurIPS 2017.- R. R. Selvaraju và cộng sự, "Grad-CAM: Visual Explanations from Deep Networks via  Gradient-based Localization", ICCV 2017.- T.-Y. Lin và cộng sự, "Focal Loss for Dense Object Detection", ICCV 2017.- Bộ dữ liệu 140k Real and Fake Faces:  https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces